The notebook is a Proof of concept of summarization of politic speeches.
It evaluates 4 models (For complet benchmark, we have to use more than 1 speech) 

In [1]:
from datetime import datetime
import hashlib
import json
from neo4j import GraphDatabase
import ollama
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import sys
import time
from tqdm import tqdm
import torch
from transformers import logging
logging.set_verbosity_error() # mask non critical error
sys.path.insert(1, "src/graph/")
# from graph_builder import (
#     get_node_id,
#     extract_graph,
#     compute_chunk_embeddings,
#     merge_graphs,
#     validate_graph,
#     add_speaker_entities,
#     build_neo4j_graph,
#     feed_global_report,
#     save_graph,
#     load_to_neo4j,
#     create_constraints
# )
sys.path.insert(1, "src/preprocessing/")
from speeches import (
    load_speeches,
    split_into_chunks#,
    # load_prompt_template
)


# Goal

The goal of the pipeline is to extract information inside a document. For the demonstration we use public speech as dataset:

 ```src/test/2017-05-14_d-claration-de-m-emmanuel-macron-pr-sident-de-la-r-publique.txt```

The extraction step will be processed by a LLM. Different models exist and the result depend on many criteria:

* Model should be multi-langage or French
* Can handle abstract concept 
* Can be executed on a home desktop in a efficient way

According to the litterature ```qwen2.5:7b``` model fit those criteria and have good results in text extraction.

# Install Ollama and the Qwen model

Before using the pipeline, install the following dependencies.

**Ollama** on Windows the install command is:

```
irm https://ollama.com/install.ps1 | iex
```

Check that the Ollama server is running:

```
ollama list
```

You can also check the API response directly:

```
curl http://localhost:11434/api/tags
```

If you've just installed Ollama, you won't have any models yet. Pull `qwen2.5:7b`:

```
ollama pull qwen2.5:7b
```

Check that it runs correctly:

```
ollama run qwen2.5:7b
```

You can type anything into the prompt to confirm it responds.

# Load speech

We load the speech and made a short analysis about it

As you can see, the file is composed of many bloc. Title, date, source, speech, keywords...
The speech is between the keywords:

* Texte intégral
* MOTS CLÉS

We will extract the speech and keep the other information. For this purpose we will use a home made function.
The function will return a list of dictionary. As we have only 1 text, we will save the dict intto ```speech```.

Here the different keys field:

* id
* date
* title
* filename
* header
* text
* speakers
* chunks

In [2]:
corpus = load_speeches(
    os.path.join(
        "src", "test", "speech")
    )

speech = corpus[0]

In [3]:
print(speech["text"])

"Mesdames, Messieurs,
Les Français ont choisi, vous l'avez rappelé, le 7 mai dernier, l'espoir et l'esprit de conquête.
Le monde entier a regardé notre élection présidentielle. Partout, on se demandait si les Français allaient décider à leur tour de se replier sur le passé illusoire, s'ils allaient rompre avec la marche du monde, quitter la scène de l'Histoire, céder à la défiance démocratique, l'esprit de division et tourner le dos aux Lumières, ou si au contraire ils allaient embrasser l'avenir, se donner collectivement un nouvel élan, réaffirmer leur foi dans les valeurs qui ont fait d'eux un grand peuple.
Le 7 mai, les Français ont choisi. Qu'ils en soient ici remerciés.
La responsabilité qu'ils m'ont confiée est un honneur, dont je mesure la gravité.
Le monde et l'Europe ont aujourd'hui, plus que jamais, besoin de la France. Ils ont besoin d'une France forte et sûre de son destin. Ils ont besoin d'une France qui porte haut la voix de la liberté et de la solidarité. Ils ont besoin 

# Summarization 

As you can see, the document is the first speech of the French president Macron.
Before extract information about it, lets do premilary analysis by doing summarization.
It creates a shorter version of a document or an article that captures all the important information.

We use Encoder-Decoder model. First step is to split the speech on different chunks. Indeed the speech is too long for our current model.
The function ```split_into_chunks()``` call ```RecursiveCharacterTextSplitter``` from ```langchain_text_splitters```.

**In the current step, we set the chunk_overlap to 0 because we want to summarize. In RAG context, we will change the value** 


In [4]:
chunks = split_into_chunks(speech, chunk_size=1200, chunk_overlap=0)

In [5]:
chunks[1]

{'speech_id': '2017-05-14_d-claration-de-m-emmanuel-macron-pr-sident-de-la-r-publique',
 'chunk_id': 1,
 'date': '2017-05-14',
 'title': 'Déclaration de M. Emmanuel Macron, Président de la République, sur les priorités de son quinquennat, à Paris le 14 mai 2017.',
 'text': "Le monde a besoin de ce que les Françaises et les Français lui ont toujours enseigné : l'audace de la liberté, l'exigence de l'égalité, la volonté de la fraternité.\nOr, depuis des décennies, la France doute d'elle-même. Elle se sent menacée dans sa culture, dans son modèle social, dans ses croyances profondes. Elle doute de ce qui l'a faite.\nVoilà pourquoi mon mandat sera guidé par deux exigences.\nLa première sera de rendre aux Français cette confiance en eux, depuis trop longtemps affaiblie. Je vous rassure, je n'ai pas pensé une seule seconde qu'elle se restaurerait comme par magie le soir du 7 mai. Ce sera un travail lent, exigeant, mais indispensable.\nIl m'appartiendra de convaincre les Françaises et les Fra

### Benchmarking models
chunk_overlap is set 0 and split speech as chunks. This vector will be use as input for the transformer to summarize the text.
As we use french text, we need a multilangual or French model. A benchmark is made with these models:  

* ```mT5_multilingual_XLSum```
* ```BARThez```
* ```CamemBERT```
* ```FlauBERT```

For that goal, lets create a function takes a instantiated tokenizer, model and prompts (instruction + chunk).

In [ ]:
import time
from typing import Union
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

@torch.inference_mode()
def summarize_batch_benchmark(
    tokenizer: AutoTokenizer, 
    model: AutoModelForSeq2SeqLM, 
    prompts: list[str], 
    device: Union[torch.device, str],
    max_new_tokens: int = 256,
    forced_bos_token_id: Union[int, None] = None,
    use_bfloat16: bool = False
):
    """Summarize a batch of text and measure generation throughput."""
    
    model.eval()
    
    # Tokenize input batch
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024
    ).to(device)

    # Clean device_type string for autocast ("cuda" or "cpu")
    device_type = "cuda" if "cuda" in str(device) else "cpu"

    # Synchronize GPU before timing
    if device_type == "cuda":
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    # Inference with Mixed Precision
    with torch.amp.autocast(
        enabled=use_bfloat16, 
        dtype=torch.bfloat16, 
        device_type=device_type
    ):
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=max_new_tokens,
            min_length=30,
            num_beams=2,
            early_stopping=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            no_repeat_ngram_size=3,
            do_sample=False
        )

    # Synchronize GPU after inference
    if device_type == "cuda":
        torch.cuda.synchronize()
    execution_time = time.perf_counter() - start_time

    # Calculate exact generated tokens (excluding padding)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    generated_tokens_count = (outputs != pad_id).sum().item()

    tokens_per_second = (generated_tokens_count / execution_time) if execution_time > 0 else 0.0

    metrics = {
        "execution_time_sec": round(execution_time, 4),
        "total_generated_tokens": generated_tokens_count,
        "tokens_per_second": round(tokens_per_second, 2),
        "batch_size": len(prompts)
    }

    summaries = tokenizer.batch_decode(
        outputs, 
        skip_special_tokens=True
    )
    return summaries, metrics

In [7]:
# The code bellow check if CUDA is avaible
# If CUDA is present, we can increase the speed
if torch.cuda.is_available():
    device = "cuda" 
else:
    device = "cpu"

#### mT5_multilingual_XLSum

In [8]:
checkpoint = "csebuetnlp/mT5_multilingual_XLSum" 
# load a T5 tokenizer to process text and summary
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# weights_only=False because the model file is old
model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    weights_only=False).to(device)

print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_mT5, metrics_mT5 = summarize_batch_benchmark(
    tokenizer=tokenizer,
    model=model,
    prompts = ["summarize: " + c["text"] for c in chunks],
    device=device
)

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

Calculs exécutés sur : CUDA


In [9]:
#Let see the first 5 chunks and the summary
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_mT5[i])

CHUNK 0
"Mesdames, Messieurs,
Les Français ont choisi, vous l'avez rappelé, le 7 mai dernier, l'espoir et l'esprit de conquête.
Le monde entier a regardé notre élection présidentielle. Partout, on se demandait si les Français allaient décider à leur tour de se replier sur le passé illusoire, s'ils allaient rompre avec la marche du monde, quitter la scène de l'Histoire, céder à la défiance démocratique, l'esprit de division et tourner le dos aux Lumières, ou si au contraire ils allaient embrasser l'avenir, se donner collectivement un nouvel élan, réaffirmer leur foi dans les valeurs qui ont fait d'eux un grand peuple.
Le 7 mai, les Français ont choisi. Qu'ils en soient ici remerciés.
La responsabilité qu'ils m'ont confiée est un honneur, dont je mesure la gravité.
Le monde et l'Europe ont aujourd'hui, plus que jamais, besoin de la France. Ils ont besoin d'une France forte et sûre de son destin. Ils ont besoin d'une France qui porte haut la voix de la liberté et de la solidarité. Ils ont

As you can see. The model cannot summary the different chunk and hallucinate.

#### BARThez

In [10]:
from transformers import BartTokenizer, AutoModelForSeq2SeqLM

barthez_ckpt = "moussaKam/barthez-orangesum-abstract"

# Chargement du tokenizer et du modèle appropriés
barthez_tokenizer = BartTokenizer.from_pretrained(barthez_ckpt)
barthez_model = AutoModelForSeq2SeqLM.from_pretrained(barthez_ckpt).to(device)

if barthez_tokenizer.pad_token is None:
    barthez_tokenizer.pad_token = barthez_tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_BARThez, metrics_BARThez = summarize_batch_benchmark(
    tokenizer = barthez_tokenizer,
    model = barthez_model,
    prompts = [c["text"] for c in chunks], #BARThez does not need a "summarize: " prefix like T5
    device=device
)

Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

Calculs exécutés sur : CUDA


Let see the first 5 chunks and the summary

In [11]:
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_BARThez[i])

CHUNK 0
"Mesdames, Messieurs,
Les Français ont choisi, vous l'avez rappelé, le 7 mai dernier, l'espoir et l'esprit de conquête.
Le monde entier a regardé notre élection présidentielle. Partout, on se demandait si les Français allaient décider à leur tour de se replier sur le passé illusoire, s'ils allaient rompre avec la marche du monde, quitter la scène de l'Histoire, céder à la défiance démocratique, l'esprit de division et tourner le dos aux Lumières, ou si au contraire ils allaient embrasser l'avenir, se donner collectivement un nouvel élan, réaffirmer leur foi dans les valeurs qui ont fait d'eux un grand peuple.
Le 7 mai, les Français ont choisi. Qu'ils en soient ici remerciés.
La responsabilité qu'ils m'ont confiée est un honneur, dont je mesure la gravité.
Le monde et l'Europe ont aujourd'hui, plus que jamais, besoin de la France. Ils ont besoin d'une France forte et sûre de son destin. Ils ont besoin d'une France qui porte haut la voix de la liberté et de la solidarité. Ils ont

#### CamemBERT

In [12]:
from transformers import MBart50TokenizerFast, AutoModelForSeq2SeqLM

camembert_ckpt = "facebook/mbart-large-50-many-to-many-mmt"

camembert_tokenizer = MBart50TokenizerFast.from_pretrained(camembert_ckpt)
# Load model to GPU
camembert_model = AutoModelForSeq2SeqLM.from_pretrained(camembert_ckpt).to(device)

# Configuration de la langue cible en Français pour forcer le modèle à répondre en français
camembert_tokenizer.src_lang = "fr_XX"
# On récupère l'identifiant du jeton français pour le donner au décodeur
forced_bos_token_id = camembert_tokenizer.lang_code_to_id["fr_XX"]

if camembert_tokenizer.pad_token is None:
    camembert_tokenizer.pad_token = camembert_tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_camembert, metrics_camembert = summarize_batch_benchmark(
    tokenizer = camembert_tokenizer,
    model = camembert_model,
    prompts = [c["text"] for c in chunks],
    device=device,
    forced_bos_token_id=forced_bos_token_id
)



Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Calculs exécutés sur : CUDA


Let see the first 5 chunks and the summary

In [13]:
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_camembert[i])

CHUNK 0
"Mesdames, Messieurs,
Les Français ont choisi, vous l'avez rappelé, le 7 mai dernier, l'espoir et l'esprit de conquête.
Le monde entier a regardé notre élection présidentielle. Partout, on se demandait si les Français allaient décider à leur tour de se replier sur le passé illusoire, s'ils allaient rompre avec la marche du monde, quitter la scène de l'Histoire, céder à la défiance démocratique, l'esprit de division et tourner le dos aux Lumières, ou si au contraire ils allaient embrasser l'avenir, se donner collectivement un nouvel élan, réaffirmer leur foi dans les valeurs qui ont fait d'eux un grand peuple.
Le 7 mai, les Français ont choisi. Qu'ils en soient ici remerciés.
La responsabilité qu'ils m'ont confiée est un honneur, dont je mesure la gravité.
Le monde et l'Europe ont aujourd'hui, plus que jamais, besoin de la France. Ils ont besoin d'une France forte et sûre de son destin. Ils ont besoin d'une France qui porte haut la voix de la liberté et de la solidarité. Ils ont

#### FlauBERT

In [14]:
from transformers import T5TokenizerFast, AutoModelForSeq2SeqLM

flaubert_ckpt = "plguillou/t5-base-fr-sum-cnndm"

flaubert_tokenizer = T5TokenizerFast.from_pretrained(flaubert_ckpt)
flaubert_model = AutoModelForSeq2SeqLM.from_pretrained(flaubert_ckpt).to(device)

if flaubert_tokenizer.pad_token is None:
    flaubert_tokenizer.pad_token = flaubert_tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_flaubert, metrics_flaubert = summarize_batch_benchmark(
    tokenizer = flaubert_tokenizer,
    model = flaubert_model,
    prompts = ["summarize: "  + c["text"] for c in chunks], 
    device=device
)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Calculs exécutés sur : CUDA


Let see the first 5 chunks and the summary

In [15]:
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_flaubert[i])

CHUNK 0
"Mesdames, Messieurs,
Les Français ont choisi, vous l'avez rappelé, le 7 mai dernier, l'espoir et l'esprit de conquête.
Le monde entier a regardé notre élection présidentielle. Partout, on se demandait si les Français allaient décider à leur tour de se replier sur le passé illusoire, s'ils allaient rompre avec la marche du monde, quitter la scène de l'Histoire, céder à la défiance démocratique, l'esprit de division et tourner le dos aux Lumières, ou si au contraire ils allaient embrasser l'avenir, se donner collectivement un nouvel élan, réaffirmer leur foi dans les valeurs qui ont fait d'eux un grand peuple.
Le 7 mai, les Français ont choisi. Qu'ils en soient ici remerciés.
La responsabilité qu'ils m'ont confiée est un honneur, dont je mesure la gravité.
Le monde et l'Europe ont aujourd'hui, plus que jamais, besoin de la France. Ils ont besoin d'une France forte et sûre de son destin. Ils ont besoin d'une France qui porte haut la voix de la liberté et de la solidarité. Ils ont

#### Model performance

* mT5 hallucinate 
* BARThez's output format is not correct
* Camembert output is troncated. The model does not summarize correctly 
* flaubert seems to give correct results

In case we have 2 or more model with good results, we can compare the number of token per seconde. Choose the most efficient is important to increase the speed process.
A lot of time is lost during the inference process (we wait the end of the procedure).

In [19]:
df_bench = pd.DataFrame(
    {    
        'tokens_per_second' : [metrics_mT5['tokens_per_second'], metrics_BARThez['tokens_per_second'], metrics_camembert['tokens_per_second'], metrics_flaubert['tokens_per_second'] ]
        
    }, index = ["mT5", "BARThez", "camembert", "flaubert" ]
)

df_bench.head()

,tokens_per_second
mT5,91.54
BARThez,278.76
camembert,5.52
flaubert,5.44


# Metric score

In the notebook I made a visual inspection. However in real life we cannot check everything. different strategies exist to evaluate the validity.

* LLM-as-a-Judge: a LLM will evaluate the text generated based on **Faithfulness**, **Relevance** and **Fluidité**.

*An example of prompt we can use:*

```"Voici un texte source : {texte_source}. Voici le résumé généré : {resume}. Note sur 5 la fidélité du résumé au texte source et donne une note de concision."```



In [ ]:
import json
import evaluate
import pandas as pd
from langchain_ollama import ChatOllama

def evaluate_with_llm(
    predictions: list[str], 
    sources: list[str], 
    model_name: str = "qwen2.5:7b"
) -> dict:
    """
    Utilise Qwen 2.5 (7B) localement pour évaluer les résumés.
    """
    # Configuration du modèle local via Ollama
    llm = ChatOllama(model=model_name, temperature=0.0)
    
    prompt_template = """Tu es un expert en évaluation de résumés automatiques.
Évalue le résumé suivant uniquement à partir du texte source fourni.

[TEXTE SOURCE]:
{source}

[RÉSUMÉ À ÉVALUER]:
{prediction}

Donne une note entière de 1 à 5 pour chacun des 3 critères suivants :
1. Fidélité : Le résumé est-il exact et exempt d'hallucinations ?
2. Couverture : Le résumé retient-il les points clés du texte source ?
3. Concision : La rédaction est-elle synthétique, claire et fluide en français ?

Réponds STRICTEMENT avec un objet JSON au format suivant, sans aucun autre texte avant ou après :
{{"fidelite": 4, "couverture": 5, "concision": 4}}
"""

    fidelite_scores = []
    couverture_scores = []
    concision_scores = []

    for src, pred in zip(sources, predictions):
        formatted_prompt = prompt_template.format(source=src, prediction=pred)
        
        try:
            response = llm.invoke(formatted_prompt).content
            
            # Extraction propre du JSON
            clean_json = response[response.find('{'):response.rfind('}')+1]
            data = json.loads(clean_json)
            
            fidelite_scores.append(data.get("fidelite", 0))
            couverture_scores.append(data.get("couverture", 0))
            concision_scores.append(data.get("concision", 0))
        except Exception as e:
            print(f"Erreur de parsing sur un résumé : {e}")

    def mean(lst): return round(sum(lst) / len(lst), 2) if lst else 0.0

    return {
        "llm_fidelite": mean(fidelite_scores),
        "llm_couverture": mean(couverture_scores),
        "llm_concision": mean(concision_scores)
    }

In [ ]:
sources = [c["text"] for c in chunks]
llm_models = {"mistral:7b-instruct" : None, "qwen2.5:7b" : None}
for judge_llm in llm_models.keys():
    results = {}
    for name, chunk_model in zip(
        ["mT5", "BARThez", "camembert", "flaubert" ],
        [chunk_summaries_mT5, chunk_summaries_BARThez, chunk_summaries_camembert, chunk_summaries_flaubert]
        ):
        judge_scores = evaluate_with_llm(
            predictions = chunk_model, 
            sources = sources,
            model_name = judge_llm
        )
        results[name] = judge_scores
        # print("Notes obtenues par "+name+" via "+judge_llm+" :")
        # print(judge_scores)
    llm_models[judge_llm] = results

In [20]:
# Update the benchmark with the new results
# Add comparison between mistal and qwen
for judge_llm in llm_models.keys():
    df_bench = pd.concat(
        [
            df_bench, 
            pd.DataFrame(llm_models[judge_llm]).T.add_suffix("_"+judge_llm)
        ],
        axis=1
    )
df_bench.sort_index(axis=1)

,llm_concision_mistral:7b-instruct,llm_concision_qwen2.5:7b,llm_couverture_mistral:7b-instruct,llm_couverture_qwen2.5:7b,llm_fidelite_mistral:7b-instruct,llm_fidelite_qwen2.5:7b,tokens_per_second
mT5,3.50,4.0,4.38,5.0,3.62,4.0,91.54
BARThez,2.75,4.0,3.00,5.0,2.88,4.0,278.76
camembert,4.00,4.0,5.00,5.0,4.00,4.0,5.52
flaubert,4.00,4.0,5.00,5.0,4.00,4.0,5.44


In [ ]:
pd.DataFrame(llm_models[judge_llm]).index.add_suffix('_'+judge_llm)

In [61]:
dfs = []
for judge_llm, data in llm_models.items():
    sub_df = pd.DataFrame(data)
    sub_df.index = sub_df.index.map(lambda x: f"{x}_{judge_llm}")
    dfs.append(sub_df)

df = pd.concat(dfs)

In [62]:
df

,mT5,BARThez,camembert,flaubert
llm_fidelite_mistral:7b-instruct,3.62,2.88,4.0,4.0
llm_couverture_mistral:7b-instruct,4.38,3.00,5.0,5.0
llm_concision_mistral:7b-instruct,3.50,2.75,4.0,4.0
llm_fidelite_qwen2.5:7b,4.00,4.00,4.0,4.0
llm_couverture_qwen2.5:7b,5.00,5.00,5.0,5.0
llm_concision_qwen2.5:7b,4.00,4.00,4.0,4.0


In [51]:
df_bench

,tokens_per_second,llm_fidelite_mistral:7b-instruct,llm_couverture_mistral:7b-instruct,llm_concision_mistral:7b-instruct,llm_fidelite_qwen2.5:7b,llm_couverture_qwen2.5:7b,llm_concision_qwen2.5:7b
mT5,12.96,3.62,5.0,4.0,3.62,5.0,4.0
BARThez,198.89,4.00,5.0,4.0,4.00,5.0,4.0
camembert,34.94,4.00,5.0,4.0,4.00,5.0,4.0
flaubert,10.01,4.00,5.0,4.0,4.00,5.0,4.0


In [ ]:
# Update the benchmark with the new results
df_bench = pd.concat([df_bench, pd.DataFrame(results).T], axis=1)
df_bench.head()

Common metrics used to evaluate text summarization:

* ROUGE (Recall-Oriented Understudy for Gisting Evaluation) counts the number of eatc word or n-gram in commun between summary and a reference text.It fast and no need LLM or GPU but cannot catch cynonym and rephrase. 

* BERTScore compute semantic similarity. It use a langage model (like camemBERT) and the scoring will depend on the model use.

In [ ]:
import evaluate

def evaluate_summaries(predictions: list[str], references: list[str]) -> dict:
    """
    Evaluate summaries if you provide references
    """
    # Load metrice
    rouge_metric = evaluate.load("rouge")
    bert_metric = evaluate.load("bertscore")

    # ROUGE
    rouge_results = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False # unable for french text
    )

    # BERTScore (with CamemBERT)
    bert_results = bert_metric.compute(
        predictions=predictions,
        references=references,
        lang="fr",
        model_type="camembert-base"
    )

    # Computes metric
    mean_bert_f1 = sum(bert_results["f1"]) / len(bert_results["f1"])

    metrics = {
        "rouge1": round(rouge_results["rouge1"], 4),
        "rouge2": round(rouge_results["rouge2"], 4),
        "rougeL": round(rouge_results["rougeL"], 4),
        "bertscore_f1": round(mean_bert_f1, 4)
    }

    return metrics

ROUGE and BERTScore expect a reference summary. As we do not have it. The final evaluation will be based on BERTScore (Source-to-Summary). We compute how close the sumary is close the orignal speech, evaluate the compression rate and lisibility (Flesch / Gunning Fog).

In [ ]:
import evaluate

def evaluate_without_reference(
    metric_model,
    model_type: str,
    predictions: list[str], 
    sources: list[str],
    batch_size: int = 16
) -> list[dict]:
    """
    Calcule la proximité sémantique sur GPU (CUDA) avec un modèle d'embedding neutre.
    """
    if torch.cuda.is_available():
        target_device = "cuda" 
    else:
        target_device = "cpu"

    results = metric_model.compute(
        predictions=predictions,
        references=sources,
        lang="fr",
        model_type=model_type,
        device=target_device,
        batch_size=batch_size
    )

    scores = []
    for i, (pred, src) in enumerate(zip(predictions, sources)):
        compression_rate = 1 - (len(pred.split()) / len(src.split()))
        scores.append({
            "semantic_coverage_f1": round(results["f1"][i], 4),
            "compression_rate": round(compression_rate, 2)
        })

    return scores

In [ ]:
sources = [c["text"] for c in chunks]
results_bertscore = {}
# load bertscore
model_metric = evaluate.load("bertscore")
# Use roberta to avoid bias (camembert evaluate camembert for example)
neutral_model_type = "xlm-roberta-base"
for name, chunk_model in zip(
    ["mT5", "BARThez", "camembert", "flaubert"],
    [chunk_summaries_mT5, chunk_summaries_BARThez, chunk_summaries_camembert, chunk_summaries_flaubert]
):
    scores = evaluate_without_reference(
        metric_model = model_metric,
        model_type = neutral_model_type,
        predictions = chunk_model, 
        sources = sources
    )
    
    avg_f1 = sum(s["semantic_coverage_f1"] for s in scores) / len(scores)
    avg_comp = sum(s["compression_rate"] for s in scores) / len(scores)
    
    results_bertscore[name] = {
        "avg_bertscore_f1": round(avg_f1, 4),
        "avg_compression_rate": round(avg_comp, 2)
    }
    
    print(f"--- {name} ---")
    print(f"BERTScore F1 moyen : {round(avg_f1, 4)}")
    print(f"Taux de compression moyen : {round(avg_comp, 2)}\n")

In [ ]:
df_bench = pd.concat([df_bench, pd.DataFrame(results_bertscore).T], axis=1)
df_bench.head()

# Conclusion

This cross-evaluation combines throughput speed (tokens/sec), LLM-assisted evaluation (Qwen 2.5), and neutral semantic similarity (XLM-RoBERTa BERTScore) to identify two distinct winners based on project constraints.

## Execution Speed (tokens/sec):
* BARThez (241.43 tok/s) clearly dominates the benchmark. Its high throughput makes it the most cost-effective choice for batch processing and real-time integration.

* CamemBERT (167.86 tok/s) maintains a solid intermediate processing speed.

* FlauBERT (65.90 tok/s) suffers from noticeable GPU latency, making it unsuitable for large-scale production deployment.

## LLM Quality Assessment (Fidelity, Coverage, Conciseness):

* All tested models achieved a perfect score in coverage (5.0/5) along with strong ratings in fidelity (4.0/5) and conciseness (4.0/5).

* Every model successfully extracted the core information from the source texts without hallucinating or altering the original meaning.

## Neutral BERTScore (XLM-RoBERTa) vs. Compression Rate:

* CamemBERT (0.9080) and FlauBERT (0.8975) show higher BERTScores due to a lower compression rate (64% to 78%), retaining more of the original text structure and phrasing.

* BARThez (0.8160) and mT5 (0.8193) achieve extreme compression (90% to 99%). Even though their outputs are reduced to essential key phrases, they still maintain a remarkably high semantic similarity score (>0.81) relative to their brevity.
